# P5-2_Experimentacion_Scores_Positivos

Versión temporal corregida del experimento **Scores Positivos**.

Esta variante usa únicamente las variables cuya **importancia media por permutación fue positiva en P5_scores_features**, calculada mediante validación cruzada repetida sobre el conjunto de desarrollo. La selección se deriva de `P5_CV_feature_importance_summary.csv` y el test 2023–2024 no participa en la selección ni en el ajuste.

Flujo:
- P4 genera las muestras temporales `thr1`.
- P5 calcula importancia de variables exclusivamente en desarrollo.
- P5-2 conserva las variables con `mean_importance > 0`.
- Train+validation 2021–2022 se unen como desarrollo.
- Los hiperparámetros se seleccionan con RepeatedStratifiedKFold 5×20.
- El test 2023–2024 solo se evalúa cuando `RUN_FINAL_TEST=True`.


In [13]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    precision_score, recall_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings('ignore')

SEED = 41
N_SPLITS = 5
N_REPEATS = 20
RUN_FINAL_TEST = True

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')

TRAIN_FILE = BASE_DIR / 'sample_temporal_train_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv'
VAL_FILE   = BASE_DIR / 'sample_temporal_val_hist2000_2020_target2021_2022_thr1_neg1_random_isolates.csv'
TEST_FILE  = BASE_DIR / 'sample_temporal_test_hist2000_2022_target2023_2024_thr1_neg1_random_isolates.csv'
IMPORTANCE_FILE = BASE_DIR / 'P5_CV_feature_importance_summary.csv'

OUTPUT_DIR = BASE_DIR
TARGET = 'connected'


In [14]:
# Selección reproducible de "scores positivos" a partir de P5
importance_df = pd.read_csv(IMPORTANCE_FILE)
required_cols = {'feature', 'mean_importance'}
if not required_cols.issubset(importance_df.columns):
    raise ValueError(f'Faltan columnas en {IMPORTANCE_FILE.name}: {required_cols - set(importance_df.columns)}')

sort_cols = ['rank'] if 'rank' in importance_df.columns else ['mean_importance']
ascending = [True] if 'rank' in importance_df.columns else [False]
positive_df = importance_df.loc[importance_df['mean_importance'] > 0].sort_values(sort_cols, ascending=ascending)
POSITIVE_FEATURES = positive_df['feature'].tolist()

if not POSITIVE_FEATURES:
    raise ValueError('P5 no produjo variables con mean_importance > 0.')

print(f'Features con score medio positivo: {len(POSITIVE_FEATURES)}')
for i, f in enumerate(POSITIVE_FEATURES, 1):
    value = float(positive_df.loc[positive_df['feature'] == f, 'mean_importance'].iloc[0])
    print(f'{i:2d}. {f:<28} mean_importance={value:.6f}')

with open(OUTPUT_DIR / 'P5-2_Scores_Positivos_features.txt', 'w', encoding='utf-8') as fh:
    for f in POSITIVE_FEATURES:
        fh.write(f + '\n')


Features con score medio positivo: 8
 1. sum_of_neighbors             mean_importance=0.105725
 2. sum_of_papers                mean_importance=0.047575
 3. sum_of_keywords              mean_importance=0.016898
 4. community_cn                 mean_importance=0.012727
 5. keywords_match               mean_importance=0.012313
 6. adamic_adar_index            mean_importance=0.007958
 7. resource_allocation          mean_importance=0.005685
 8. jaccard_coefficient          mean_importance=0.000153


In [15]:
train_df = pd.read_csv(TRAIN_FILE)
val_df = pd.read_csv(VAL_FILE)
test_df = pd.read_csv(TEST_FILE)

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    missing = [c for c in POSITIVE_FEATURES + [TARGET] if c not in df.columns]
    if missing:
        raise ValueError(f'{name}: faltan columnas {missing}')
    print(f"{name}: n={len(df)}, clases={df[TARGET].value_counts().sort_index().to_dict()}")

# Desarrollo = train + validation del horizonte 2021-2022
dev_df = pd.concat([train_df, val_df], ignore_index=True)
X_dev = dev_df[POSITIVE_FEATURES].copy()
y_dev = dev_df[TARGET].astype(int).to_numpy()

X_test = test_df[POSITIVE_FEATURES].copy()
y_test = test_df[TARGET].astype(int).to_numpy()

print(f"desarrollo train+val: n={len(dev_df)}, clases={pd.Series(y_dev).value_counts().sort_index().to_dict()}")
print(f"test reservado: n={len(test_df)}, clases={pd.Series(y_test).value_counts().sort_index().to_dict()}")


train: n=27, clases={0: 13, 1: 14}
val: n=7, clases={0: 4, 1: 3}
test: n=112, clases={0: 56, 1: 56}
desarrollo train+val: n=34, clases={0: 17, 1: 17}
test reservado: n=112, clases={0: 56, 1: 56}


## Selección de hiperparámetros solo en desarrollo

El imputador y el escalador se ajustan dentro de cada fold. La métrica principal para seleccionar hiperparámetros es AUC-PR.

In [16]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

model_spaces = {
    'Decision Tree': (
        DecisionTreeClassifier(random_state=SEED),
        {
            'model__criterion': ['gini', 'entropy'],
            'model__max_depth': [2, 4, None],
            'model__min_samples_leaf': [1, 2, 4],
        }
    ),
    'SVM - Linear Kernel': (
        LinearSVC(random_state=SEED),
        {
            'model__C': [0.01, 0.1, 1.0, 10.0],
            'model__dual': [False],
        }
    ),
    'SVM - RBF Kernel': (
        SVC(kernel='rbf', probability=True, random_state=SEED),
        {
            'model__C': [0.1, 1.0, 5.0, 10.0],
            'model__gamma': ['scale', 'auto'],
        }
    ),
    'K-Neighbors': (
        KNeighborsClassifier(),
        {
            'model__n_neighbors': [3, 5, 7, 11],
            'model__weights': ['uniform', 'distance'],
            'model__p': [1, 2],
        }
    ),
    'Naive Bayes': (
        GaussianNB(),
        {
            'model__var_smoothing': [1e-11, 1e-9, 1e-7, 1e-5],
        }
    ),
    'Multi-Layer Perceptron': (
        MLPClassifier(max_iter=1000, random_state=SEED),
        {
            'model__hidden_layer_sizes': [(25,), (50,)],
            'model__alpha': [0.0001, 0.001],
            'model__learning_rate_init': [0.001, 0.01],
        }
    ),
}

searches = {}
summary_rows = []
best_params = {}

for model_name, (estimator, grid) in model_spaces.items():
    print()
    print(f'Buscando hiperparámetros: {model_name}')

    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', estimator),
    ])

    search = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring={
            'auc_pr': 'average_precision',
            'auc_roc': 'roc_auc',
            'f1': 'f1',
            'accuracy': 'accuracy',
        },
        refit='auc_pr',
        cv=cv,
        n_jobs=-1,
        return_train_score=False,
    )
    search.fit(X_dev, y_dev)
    searches[model_name] = search

    bp = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
    best_params[model_name] = bp

    i = search.best_index_
    row = {
        'model': model_name,
        'best_mean_auc_pr_cv': search.cv_results_['mean_test_auc_pr'][i],
        'std_auc_pr_cv': search.cv_results_['std_test_auc_pr'][i],
        'best_mean_auc_roc_cv': search.cv_results_['mean_test_auc_roc'][i],
        'std_auc_roc_cv': search.cv_results_['std_test_auc_roc'][i],
        'best_mean_f1_cv': search.cv_results_['mean_test_f1'][i],
        'std_f1_cv': search.cv_results_['std_test_f1'][i],
        'best_mean_accuracy_cv': search.cv_results_['mean_test_accuracy'][i],
        'std_accuracy_cv': search.cv_results_['std_test_accuracy'][i],
    }
    summary_rows.append(row)
    print('Mejores parámetros:', bp)
    print(f"AUC-PR CV = {row['best_mean_auc_pr_cv']:.4f} +/- {row['std_auc_pr_cv']:.4f}")

cv_summary = pd.DataFrame(summary_rows).sort_values('best_mean_auc_pr_cv', ascending=False).reset_index(drop=True)
print()
print('Resumen CV de P5-2_Experimentacion_Scores_Positivos:')
print(cv_summary.to_string(index=False))



Buscando hiperparámetros: Decision Tree
Mejores parámetros: {'criterion': 'gini', 'max_depth': 2, 'min_samples_leaf': 4}
AUC-PR CV = 0.6833 +/- 0.1611

Buscando hiperparámetros: SVM - Linear Kernel
Mejores parámetros: {'C': 0.01, 'dual': False}
AUC-PR CV = 0.8059 +/- 0.1626

Buscando hiperparámetros: SVM - RBF Kernel
Mejores parámetros: {'C': 0.1, 'gamma': 'scale'}
AUC-PR CV = 0.7713 +/- 0.1854

Buscando hiperparámetros: K-Neighbors
Mejores parámetros: {'n_neighbors': 11, 'p': 1, 'weights': 'distance'}
AUC-PR CV = 0.7818 +/- 0.1632

Buscando hiperparámetros: Naive Bayes
Mejores parámetros: {'var_smoothing': 1e-05}
AUC-PR CV = 0.7710 +/- 0.1591

Buscando hiperparámetros: Multi-Layer Perceptron
Mejores parámetros: {'alpha': 0.0001, 'hidden_layer_sizes': (25,), 'learning_rate_init': 0.001}
AUC-PR CV = 0.7808 +/- 0.1566

Resumen CV de P5-2_Experimentacion_Scores_Positivos:
                 model  best_mean_auc_pr_cv  std_auc_pr_cv  best_mean_auc_roc_cv  std_auc_roc_cv  best_mean_f1_cv  st

In [17]:
# Guardar resultados de selección antes de abrir test
cv_summary.to_csv(OUTPUT_DIR / 'P5-2_Scores_Positivos_CV_model_metrics.csv', index=False)

with open(OUTPUT_DIR / 'P5-2_Scores_Positivos_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)

audit = pd.DataFrame([{
    'experiment': 'P5-2_Experimentacion_Scores_Positivos',
    'sampling': 'balanced_1_to_1',
    'threshold_coauthored_papers': 1,
    'feature_rule': 'P5 mean_importance > 0',
    'n_positive_features': len(POSITIVE_FEATURES),
    'positive_features': '|'.join(POSITIVE_FEATURES),
    'development_scope': 'train_plus_validation_2021_2022',
    'hyperparameter_selection_scope': f'RepeatedStratifiedKFold_{N_SPLITS}x{N_REPEATS}_development_only',
    'preprocessing_scope': 'inside_each_CV_training_fold',
    'test_used_for_selection': False,
    'run_final_test': RUN_FINAL_TEST,
    'n_train_original': len(train_df),
    'n_val_original': len(val_df),
    'n_development': len(dev_df),
    'n_test': len(test_df),
    'seed': SEED,
}])
audit.to_csv(OUTPUT_DIR / 'P5-2_Scores_Positivos_audit.csv', index=False)

print()
print('Archivos guardados:')
print(' - P5-2_Scores_Positivos_features.txt')
print(' - P5-2_Scores_Positivos_CV_model_metrics.csv')
print(' - P5-2_Scores_Positivos_best_params.json')
print(' - P5-2_Scores_Positivos_audit.csv')



Archivos guardados:
 - P5-2_Scores_Positivos_features.txt
 - P5-2_Scores_Positivos_CV_model_metrics.csv
 - P5-2_Scores_Positivos_best_params.json
 - P5-2_Scores_Positivos_audit.csv


## Congelación antes del test

Ejecutar primero con `RUN_FINAL_TEST=False`. Revisar y congelar la selección de variables y los hiperparámetros. Para la evaluación final, cambiar únicamente `RUN_FINAL_TEST=True` y volver a ejecutar todo el notebook.

In [18]:
def decision_score(model, X):
    if hasattr(model, 'predict_proba'):
        return np.asarray(model.predict_proba(X))[:, 1]
    if hasattr(model, 'decision_function'):
        return np.asarray(model.decision_function(X)).reshape(-1)
    return np.asarray(model.predict(X)).reshape(-1)

if RUN_FINAL_TEST:
    test_rows = []
    prediction_table = test_df[['source', 'target', TARGET]].copy() if {'source','target'}.issubset(test_df.columns) else pd.DataFrame({TARGET: y_test})

    for model_name, search in searches.items():
        final_model = search.best_estimator_
        y_pred = final_model.predict(X_test)
        y_score = decision_score(final_model, X_test)

        row = {
            'model': model_name,
            'accuracy_test': accuracy_score(y_test, y_pred),
            'precision_test': precision_score(y_test, y_pred, zero_division=0),
            'recall_test': recall_score(y_test, y_pred, zero_division=0),
            'f1_test': f1_score(y_test, y_pred, zero_division=0),
            'auc_roc_test': roc_auc_score(y_test, y_score),
            'auc_pr_test': average_precision_score(y_test, y_score),
        }
        test_rows.append(row)
        prediction_table[f'{model_name}__pred'] = y_pred
        prediction_table[f'{model_name}__score'] = y_score

    test_metrics = pd.DataFrame(test_rows).sort_values('auc_pr_test', ascending=False).reset_index(drop=True)
    print()
    print('RESULTADOS FINALES P5-2 EN TEST 2023-2024:')
    print(test_metrics.to_string(index=False))

    test_metrics.to_csv(OUTPUT_DIR / 'P5-2_Scores_Positivos_FINAL_test_metrics.csv', index=False)
    prediction_table.to_csv(OUTPUT_DIR / 'P5-2_Scores_Positivos_FINAL_test_predictions.csv', index=False)

    audit.loc[0, 'run_final_test'] = True
    audit.to_csv(OUTPUT_DIR / 'P5-2_Scores_Positivos_audit.csv', index=False)

    print()
    print('Archivos finales guardados:')
    print(' - P5-2_Scores_Positivos_FINAL_test_metrics.csv')
    print(' - P5-2_Scores_Positivos_FINAL_test_predictions.csv')
else:
    print()
    print('RUN_FINAL_TEST=False: test 2023-2024 NO fue evaluado.')



RESULTADOS FINALES P5-2 EN TEST 2023-2024:
                 model  accuracy_test  precision_test  recall_test  f1_test  auc_roc_test  auc_pr_test
           Naive Bayes       0.669643        0.880000     0.392857 0.543210      0.746173     0.787035
Multi-Layer Perceptron       0.678571        0.717391     0.589286 0.647059      0.722577     0.730614
   SVM - Linear Kernel       0.589286        0.678571     0.339286 0.452381      0.687181     0.727474
           K-Neighbors       0.625000        0.694444     0.446429 0.543478      0.606505     0.587339
         Decision Tree       0.598214        0.641026     0.446429 0.526316      0.599171     0.570499
      SVM - RBF Kernel       0.580357        0.666667     0.321429 0.433735      0.317761     0.395867

Archivos finales guardados:
 - P5-2_Scores_Positivos_FINAL_test_metrics.csv
 - P5-2_Scores_Positivos_FINAL_test_predictions.csv
